# Breakout DQN smoke test (Google Colab)

[Open this notebook in Google Colab](https://colab.research.google.com/github/benphuang/atari/blob/main/atari_smoke_test.ipynb).

In Colab, select **Runtime → Change runtime type → T4 GPU** (or another available GPU) before running. The notebook installs Atari dependencies into Colab's active Python, gets the project from its public GitHub repository, and runs a short Breakout smoke test. It does not use the cluster `.venv` and contains no shell or bash cells. Colab GPU availability and memory vary; the DQN minibatch is small and replay stores individual frames to limit memory use.

**ROM terms:** ROM installation is a separate, interactive step. Read and accept the AutoROM license before entering `ACCEPT` in that cell. Training is not started automatically by opening the notebook; run the cells in order when ready. Colab's `/content` storage is temporary.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/benphuang/atari.git"
PROJECT_DIR = Path("/content/atari")

if (PROJECT_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
elif PROJECT_DIR.exists():
    raise FileExistsError(f"{PROJECT_DIR} exists but is not a git checkout; remove or rename it and retry.")
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)], check=True)

# Colab supplies NumPy and PyTorch; leave its preinstalled builds intact.
packages = [
    "ale-py>=0.11",
    "AutoROM>=0.6",
    "gymnasium>=1.0,<2",
    "opencv-python-headless>=4.8",
    "tensorboard>=2.14",
    "tyro>=0.8",
]
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *packages], check=True)

import torch
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    print(f"GPU: {torch.cuda.get_device_name(0)} ({free_bytes / 2**30:.1f} GiB free / {total_bytes / 2**30:.1f} GiB total)")
else:
    print("GPU unavailable; the smoke run will use CPU.")

## Install Atari ROMs (requires your explicit license acceptance)

This step downloads ROMs using AutoROM. Review its license and run this cell only if you accept its terms. The package was installed without the auto-accepting extra so consent is not implied by dependency installation.

In [ ]:
import shutil

license_response = input("After reviewing the AutoROM license, type ACCEPT to download ROMs: ").strip()
if license_response != "ACCEPT":
    raise RuntimeError("ROM installation skipped. Review the license, then rerun this cell and type ACCEPT if you agree.")

autorom = shutil.which("AutoROM")
if autorom is None:
    raise RuntimeError("AutoROM executable was not installed in this runtime.")
subprocess.run([autorom, "--accept-license"], check=True)

In [ ]:
import os
import sys

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))

import ale_py
import gymnasium as gym

gym.register_envs(ale_py)
env = gym.make("BreakoutNoFrameskip-v4")
observation, info = env.reset(seed=1)
assert observation.ndim == 3 and observation.shape[-1] == 3, f"Unexpected Atari observation: {observation.shape}"
print(f"Breakout preflight passed: observation shape={observation.shape}, actions={env.action_space.n}")
env.close()

## Run the short Breakout smoke test

This uses the `arxiv2013` preset with a 2,000-step cap, reduced replay capacity, and periodic evaluations disabled. The trainer still performs its final evaluation and writes TensorBoard logs under `runs/`.

In [ ]:
import os
import runpy
import sys
from pathlib import Path

import torch

PROJECT_DIR = Path("/content/atari").resolve()
TRAINING_SCRIPT = PROJECT_DIR / "atari_mnih.py"
if not TRAINING_SCRIPT.is_file():
    raise FileNotFoundError(f"{TRAINING_SCRIPT} is missing. Run the setup cell first to clone the project.")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
os.chdir(PROJECT_DIR)

smoke_args = [
    "--preset", "arxiv2013",
    "--exp-name", "colab_breakout_smoke",
    "--seed", "1",
    "--env-id", "BreakoutNoFrameskip-v4",
    "--total-timesteps", "2000",
    "--buffer-size", "5000",
    "--batch-size", "32",
    "--learning-starts", "500",
    "--exploration-steps", "2000",
    "--train-frequency", "4",
    "--eval-frequency", "0",
    "--eval-steps", "250",
    "--heldout-frequency", "0",
    "--replay-storage", "frames",
    "--noop-max", "10",
]
if not torch.cuda.is_available():
    smoke_args.append("--no-cuda")

print(f"Training device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
previous_argv = sys.argv
try:
    sys.argv = [str(TRAINING_SCRIPT), *smoke_args]
    runpy.run_path(str(TRAINING_SCRIPT), run_name="__main__")
finally:
    sys.argv = previous_argv

In [ ]:
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

event_files = sorted(Path("runs").glob("BreakoutNoFrameskip-v4__colab_breakout_smoke__1__*/events.out.tfevents.*"))
assert event_files, "No TensorBoard event file was written for the smoke run."
events = EventAccumulator(str(event_files[-1])).Reload()
assert "eval/mean_return" in events.Tags()["scalars"], f"Missing final evaluation metric; tags: {events.Tags()}"
evaluation = events.Scalars("eval/mean_return")[-1]
assert evaluation.step == 2000, f"Expected final evaluation at step 2000; got step {evaluation.step}"
print(f"Smoke test passed: final Breakout mean return={evaluation.value:.2f} at step {evaluation.step}")